In [ ]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [ ]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [ ]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

## Combinar enunciados con clasificaciones

In [ ]:
def combine_exercises(csv_classified, csv_enun = "ejercicios_clasificados.csv"): 
    df_enun = pd.read_csv(root / "csv" / csv_enun, encoding="utf-8")
    
    # set index id
    df_enun = df_enun.set_index("id")

    # load classified exercises to be merged
    df_classified = pd.read_csv(root / "csv" / csv_classified)
    
    # set index id
    df_classified = df_classified.set_index("id")

    # merge classified (df_classified) with already classified exercises (df_enun)
    # check if df_classified contains column observaciones
    if "observaciones" in df_classified.columns:
        df_enun.update(df_classified.loc[:, ["tipo_ejercicio", "observaciones"]])    
    else:
        df_enun.update(df_classified.tipo_ejercicio)

    # restore id to a column
    df_enun = df_enun.reset_index(drop = False)  

    # rearrange columns
    rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','observaciones', 'enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
    df_enun = df_enun[rearranged_cols]

    # save merged df_enun 
    df_enun.to_csv(root / "csv/ejercicios_clasificados.csv",
                   index = False, encoding="utf-8")

    # check if there are exercises with more than
    # one type of exercise
    if any(df_enun.tipo_ejercicio.str.contains("|")):
        df_enun.tipo_ejercicio = df_enun.tipo_ejercicio.str.split("|")

        # explode column tipo_ejercicio
        df_enun = df_enun.explode("tipo_ejercicio")

        # reset index
        df_enun.reset_index(inplace=True, drop = True)
        # look for rows with duplicated id 
        duplicated_indexes = df_enun[df_enun.duplicated(subset = ["id"], keep="first")].index

        # set the new id value for exercises with multiple exercise types
        for idx in duplicated_indexes:
            # get tipo_ejercicio
            tipo_ejercicio = df_enun.loc[idx].tipo_ejercicio

            # clean tipo_ejercicio prior to be inserted in id string
            add_to_id = remove_accents(tipo_ejercicio).replace(" ", "_").lower()

            # get id value
            id = df_enun.loc[idx].id

            # find position of year in id string
            year_pos = id.find("2")
    
            # compose the new id string
            new_id = id[:year_pos] + add_to_id + "_" + id[year_pos:]

            # finally, set the new id value
            df_enun.loc[idx, "id"] = new_id

    return df_enun                          

In [ ]:
# f = combine_exercises("ejercicios_clasificados_funciones.csv")

In [ ]:
# f[f.tema == "funciones"].sample(5)

In [ ]:
# f[(f.tema == "funciones") & (f.tipo_ejercicio == "optimización")]

Guardar ejercicios clasificados

In [ ]:
# f.to_csv(root / "csv/ejercicios_clasificados.csv", index = False, encoding="utf-8")